
# Lecture 6 — Python CGI: Hands‑on Exercises

These exercises cover the key concepts from the Lecture 6 slides: HTTP vs WWW, URLs, headers & MIME types, HTML forms (GET vs POST), Python CGI basics (`cgi`, `cgitb`), safe subprocess, temporary files, and simple scraping.

> **Note:** Running an actual web server/CGI in this hosted notebook environment isn't practical.  
> We'll simulate common CGI tasks (parsing query strings, building headers, etc.) in regular Python.



## Exercise 1 — URL Anatomy with `urllib.parse`

Write a function `analyze_url(url: str)` that returns a dict with keys:
- `scheme` (e.g. `"https"`)
- `host` (netloc)
- `path` (string path part)
- `query_params` (a dict mapping keys to lists of values)

Test it with:
```python
test_url = "https://example.org:8443/a/b/index.html?gene=TP53&gene=BRCA1&limit=50"
```


In [ ]:

from urllib.parse import urlparse, parse_qs

# TODO: implement analyze_url(url: str) → dict
def analyze_url(url: str):
    raise NotImplementedError("Implement me")

# Quick tests (you can edit/extend these)
test_url = "https://example.org:8443/a/b/index.html?gene=TP53&gene=BRCA1&limit=50"
try:
    info = analyze_url(test_url)
    print(info)
except NotImplementedError as e:
    print(e)



<details>
  <summary><strong>View answer</strong></summary>

```python
from urllib.parse import urlparse, parse_qs

def analyze_url(url: str):
    parts = urlparse(url)
    return {
        "scheme": parts.scheme,
        "host": parts.netloc,
        "path": parts.path,
        "query_params": parse_qs(parts.query),  # values are lists
    }

# Demo
test_url = "https://example.org:8443/a/b/index.html?gene=TP53&gene=BRCA1&limit=50"
analyze_url(test_url)
```
</details>



## Exercise 2 — Minimal HTTP Response for CGI

Write a function `render_html_page(title, body_html)` that returns a **single string**:
- First line: `Content-Type: text/html`
- A blank line
- A minimal HTML document using the given `title` and `body_html`.

Example usage:
```python
print(render_html_page("Hello", "<h1>Hi there</h1>"))
```


In [ ]:

# TODO: implement render_html_page(title, body_html) → str
def render_html_page(title: str, body_html: str) -> str:
    raise NotImplementedError("Implement me")

# Demo (uncomment when done)
# print(render_html_page("Hello", "<h1>Hi there</h1>"))



<details>
  <summary><strong>View answer</strong></summary>

```python
def render_html_page(title: str, body_html: str) -> str:
    return (
        "Content-Type: text/html\n\n"
        f"<!DOCTYPE html>\n"
        f"<html><head><meta charset='utf-8'><title>{title}</title></head>"
        f"<body>{body_html}</body></html>"
    )

print(render_html_page("Hello", "<h1>Hi there</h1>"))
```
</details>



## Exercise 3 — Simulate GET form parsing

Given a `QUERY_STRING` (as in environment variables for CGI), parse it into a dict where each key maps to a list of values.

Implement: `parse_query_string(qs: str) -> dict[str, list[str]]`.

Test with:
```python
qs = "title=Report&gene=BRCA1&gene=TP53&limit=25"
```


In [ ]:

from urllib.parse import parse_qs

# TODO: implement parse_query_string
def parse_query_string(qs: str):
    raise NotImplementedError("Implement me")

# Demo
qs = "title=Report&gene=BRCA1&gene=TP53&limit=25"
try:
    print(parse_query_string(qs))
except NotImplementedError as e:
    print(e)



<details>
  <summary><strong>View answer</strong></summary>

```python
from urllib.parse import parse_qs

def parse_query_string(qs: str):
    return parse_qs(qs, keep_blank_values=True, strict_parsing=False)

parse_query_string("title=Report&gene=BRCA1&gene=TP53&limit=25")
```
</details>



## Exercise 4 — Simulate POST (urlencoded) body parsing

Implement `parse_post_urlencoded(body_bytes: bytes)` that returns a dict mapping keys to lists of values, assuming `application/x-www-form-urlencoded`.

Use `urllib.parse.parse_qs`. Remember to decode bytes (assume UTF‑8).

Test with:
```python
body = b"title=Hello+World&notes=alpha&notes=beta"
```


In [ ]:

from urllib.parse import parse_qs

# TODO: implement parse_post_urlencoded
def parse_post_urlencoded(body_bytes: bytes):
    raise NotImplementedError("Implement me")

# Demo
body = b"title=Hello+World&notes=alpha&notes=beta"
try:
    print(parse_post_urlencoded(body))
except NotImplementedError as e:
    print(e)



<details>
  <summary><strong>View answer</strong></summary>

```python
from urllib.parse import parse_qs

def parse_post_urlencoded(body_bytes: bytes):
    text = body_bytes.decode("utf-8", errors="replace")
    return parse_qs(text, keep_blank_values=True)

parse_post_urlencoded(b"title=Hello+World&notes=alpha&notes=beta")
```
</details>



## Exercise 5 — Safe subprocess helper

Write `run_cmd(args, timeout=5)` that executes a command **without** `shell=True`, returns decoded stdout (UTF-8), and raises a clear `RuntimeError` on non-zero exit or timeout.

Example:
```python
run_cmd(["python3", "-c", "print('ok')"])
```


In [ ]:

import subprocess

# TODO: implement run_cmd(args: list[str], timeout: int=5) → str
def run_cmd(args, timeout=5) -> str:
    raise NotImplementedError("Implement me")

# Demo (uncomment when implemented)
# print(run_cmd(["python3", "-c", "print('ok')"]))



<details>
  <summary><strong>View answer</strong></summary>

```python
import subprocess

def run_cmd(args, timeout=5) -> str:
    try:
        out = subprocess.check_output(args, timeout=timeout)
        return out.decode("utf-8", errors="replace")
    except subprocess.TimeoutExpired as e:
        raise RuntimeError(f"Command timed out after {timeout}s: {args}") from e
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Command failed ({e.returncode}): {args}\n{e.output.decode('utf-8', errors='replace')}") from e

print(run_cmd(["python3", "-c", "print('ok')"]))
```
</details>



## Exercise 6 — Unique temp filenames

Write `tmpname(prefix=\"out\", ext=\".txt\")` that returns a unique filename incorporating the process ID. Use `os.getpid()` and also add a `uuid4()` for extra safety.

Example return: `out_12345_abcd1234.txt`


In [ ]:

import os, uuid

# TODO: implement tmpname(prefix="out", ext=".txt")
def tmpname(prefix="out", ext=".txt"):
    raise NotImplementedError("Implement me")

# Demo (uncomment when done)
# print(tmpname())



<details>
  <summary><strong>View answer</strong></summary>

```python
import os, uuid

def tmpname(prefix="out", ext=".txt"):
    pid = os.getpid()
    token = uuid.uuid4().hex[:8]
    if not ext.startswith("."):
        ext = "." + ext
    return f"{prefix}_{pid}_{token}{ext}"

tmpname()
```
</details>



## Exercise 7 — Extract links from HTML (fragile scraping)

Given an HTML string, extract all links as `(href, text)` tuples. Use Python's standard `html.parser` (subclass `HTMLParser`).

**Note:** This is intentionally simple and therefore *fragile* — real pages change markup.

Use the sample:
```python
html_doc = '''
<!doctype html><html><body>
  <a href="/gene/TP53">TP53 tumor suppressor</a>
  <a href="https://example.org/docs/report.pdf">Report PDF</a>
</body></html>
'''
```


In [ ]:

from html.parser import HTMLParser

# TODO: implement LinkExtractor returning list of (href, text) tuples
class LinkExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self._in_a = False
        self._href = None
        self._buf = []
        self.links = []

    # Implement handle_starttag, handle_endtag, handle_data

# Demo
html_doc = '''
<!doctype html><html><body>
  <a href="/gene/TP53">TP53 tumor suppressor</a>
  <a href="https://example.org/docs/report.pdf">Report PDF</a>
</body></html>
'''
parser = LinkExtractor()
parser.feed(html_doc)
parser.links  # Expect: [('/gene/TP53', 'TP53 tumor suppressor'), ('https://example.org/docs/report.pdf', 'Report PDF')]



<details>
  <summary><strong>View answer</strong></summary>

```python
from html.parser import HTMLParser

class LinkExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self._in_a = False
        self._href = None
        self._buf = []
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            self._in_a = True
            self._href = dict(attrs).get("href")

    def handle_data(self, data):
        if self._in_a:
            self._buf.append(data)

    def handle_endtag(self, tag):
        if tag.lower() == "a" and self._in_a:
            text = "".join(self._buf).strip()
            if self._href:
                self.links.append((self._href, text))
            self._in_a = False
            self._href = None
            self._buf = []

# Demo
html_doc = '''
<!doctype html><html><body>
  <a href="/gene/TP53">TP53 tumor suppressor</a>
  <a href="https://example.org/docs/report.pdf">Report PDF</a>
</body></html>
'''
parser = LinkExtractor()
parser.feed(html_doc)
parser.links
```
</details>



## Exercise 8 — Guess MIME type from filename

Use Python's `mimetypes` module to implement `guess_mime(filename) -> str` that returns a MIME type string (default to `"application/octet-stream"` if unknown).

Test with: `"index.html"`, `"data.json"`, `"image.jpeg"`, `"weird.extension"`.


In [ ]:

import mimetypes

# TODO: implement guess_mime
def guess_mime(filename: str) -> str:
    raise NotImplementedError("Implement me")

# Demo (uncomment when done)
# for fn in ["index.html", "data.json", "image.jpeg", "weird.extension"]:
#     print(fn, "→", guess_mime(fn))



<details>
  <summary><strong>View answer</strong></summary>

```python
import mimetypes

def guess_mime(filename: str) -> str:
    typ, _ = mimetypes.guess_type(filename)
    return typ or "application/octet-stream"

for fn in ["index.html", "data.json", "image.jpeg", "weird.extension"]:
    print(fn, "→", guess_mime(fn))
```
</details>



## Exercise 9 — Sketch a minimal CGI script (echo form data)

Write a standalone Python CGI script that:
1. Prints `Content-Type: text/html` then a blank line.
2. Uses `cgi.FieldStorage()` to read inputs.
3. Renders a tiny HTML page that echoes a field called `name` (default: "Anonymous").
4. Includes `cgitb.enable()` at the top for debugging.

*(You do not need to execute it here; just draft the script.)*



<details>
  <summary><strong>View answer</strong></summary>

```python
#!/usr/bin/env python3
import cgi, cgitb
cgitb.enable()  # helpful tracebacks in the browser

print("Content-Type: text/html")
print()
form = cgi.FieldStorage()
name = form.getvalue("name", "Anonymous")

print(f\"\"\"<!doctype html>
<html><head><meta charset="utf-8"><title>Hello</title></head>
<body>
  <h1>Hello, {name}!</h1>
  <p>This page was generated via Python CGI.</p>
</body></html>\"\"\")
```
</details>



## Exercise 10 — GET vs POST: when and why?

You need to send a 2 KB comment string and a confidential token to the server. Which method is more appropriate: **GET** or **POST**? Explain briefly.



<details>
  <summary><strong>View answer</strong></summary>

**POST.** GET encodes data in the URL (size-limited, logged/bookmarkable), while POST places data in the request body, supports larger payloads, and avoids exposing credentials in the URL. Always use HTTPS for confidentiality.
</details>



---

✅ That’s it! If you want more practice, try extending the helpers to emit other MIME types, or adapt the link extractor to handle relative URLs properly.
